# MERFISH 500-Gene Panel Compression

This notebook asks: which of the 500 already-measured ABC MERFISH genes carry the most spatial and cell-type information when compressed to 10, 25, 50, or 100 genes?

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path("..").resolve()
sys.path.insert(0, str(PROJECT_DIR))

from src.notebook import (
    evaluate_panels,
    learned_panels,
    load_merfish,
    merge_panel_dicts,
    run_baselines,
    train_panel_sizes,
)
from src.visualization.plots import plot_panel_comparison

In [ ]:
# Edit these for your machine / experiment.
DATA_ROOT = Path("/path/to/abc_atlas")
RESULTS_DIR = PROJECT_DIR / "results" / "merfish_compression"

PANEL_SIZES = [10, 25, 50, 100]
CELLTYPE_KEY = "subclass"
BRAIN_SECTION = None  # example: "C57BL6J-638850.38"
MAX_CELLS = 50_000    # set to None for all cells if you have enough memory/time
SEED = 0

## Load MERFISH Data

The loader uses the ABC tutorial's log2 MERFISH h5ad, joins cell metadata, removes blank codewords, and keeps the real measured genes as the compression universe.

In [ ]:
adata = load_merfish(
    DATA_ROOT,
    celltype_key=CELLTYPE_KEY,
    brain_section=BRAIN_SECTION,
    max_cells=MAX_CELLS,
    seed=SEED,
)
adata

## Baseline Panels

In [ ]:
baseline_panels = run_baselines(
    adata,
    panel_sizes=PANEL_SIZES,
    celltype_key=CELLTYPE_KEY,
    include_spapros=False,
    output_dir=RESULTS_DIR / "baselines" / "spapros",
    random_seeds=[0, 1, 2],
)
baseline_panels.keys()

## Learned Panels

In [ ]:
runs = train_panel_sizes(
    adata,
    panel_sizes=PANEL_SIZES,
    output_dir=RESULTS_DIR / "trained_models",
    n_epochs=50,
    batch_size=512,
    num_workers=0,
    seed=SEED,
)

learned_panel_dict = learned_panels(runs)
{k: v["selected_gene_symbols"] for k, v in runs.items()}

## Evaluate

In [ ]:
all_panels = merge_panel_dicts(baseline_panels, learned_panel_dict)
results = evaluate_panels(
    adata,
    all_panels,
    celltype_key="ct_label",
    run_clustering_min_k=25,
    output_csv=RESULTS_DIR / "eval_summary.csv",
)
results

## Compare Metrics

In [ ]:
metrics = [m for m in ["f1_macro", "knn_overlap_mean", "nmi_mean"] if m in results.columns]
fig = plot_panel_comparison(
    results,
    metrics=metrics,
    save_path=str(RESULTS_DIR / "metric_comparison.png"),
)
fig